# ARIA - LITE

ARIA Lite is a lightweight GraphRAG-based biomedical research assistant focused on breast cancer AI literature.

The project combines two retrieval paradigms:

1. Semantic Retrieval
   Dense vector embeddings are used to retrieve papers semantically related to a user query.

2. Graph-Based Retrieval
   Biomedical entities such as genes and drugs are extracted from papers and represented as relationships in a lightweight knowledge graph.

By combining these two approaches, the system aims to provide more grounded and explainable retrieval compared to traditional vector-only RAG systems.

The project is intentionally scoped for rapid iteration and learning:
- ~300-500 PubMed papers
- Abstract-only corpus
- Lightweight graph construction
- Citation-grounded responses

Core technologies:
- PubMed / Entrez API
- SciSpacy
- Sentence Transformers
- FAISS
- Python + Google Colab

End Goal:
Build a small but functional biomedical GraphRAG system capable of retrieving relevant breast cancer AI papers and generating grounded answers with PMID citations.

# 3B_entity_linking.ipynb

PURPOSE: This notebook performs biomedical entity linking
over previously extracted entities.


INPUT: clean_papers_with_entities_raw.json


OUTPUT: clean_papers_with_linked_entities.json


WHY SEPARATE NOTEBOOK?
- avoids Colab RAM crashes
- separates NER from linking
- cleaner pipeline architecture
- easier debugging

In [1]:
# ============================================================
# SECTION 1 — Install Libraries
# ============================================================

!pip install spacy scispacy -q

!pip install \
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz -q

  Preparing metadata (setup.py) ... done


In [2]:
# ============================================================
# SECTION 2 — Imports
# ============================================================

from google.colab import drive

import os
import json
import re
from collections import defaultdict

import spacy
from scispacy.linking import EntityLinker

In [3]:
# ============================================================
# SECTION 3 — Mount Drive
# ============================================================

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ============================================================
# SECTION 4 — Paths
# ============================================================

PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2"

INPUT_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "clean_papers_with_entities.json"
)

OUTPUT_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "clean_papers_with_linked_entities.json"
)

In [5]:
# ============================================================
# SECTION 5 — Load Extracted Entities
# ============================================================

with open(INPUT_PATH, "r") as f:
    paper_entities = json.load(f)

print("Loaded papers:", len(paper_entities))

Loaded papers: 1079


In [6]:
# ============================================================
# SECTION 6 — Load Linking Model
# ============================================================

print("Loading SciSpacy linker...")

linker_nlp = spacy.load("en_core_sci_md")

linker_nlp.add_pipe(
    "scispacy_linker",
    config={
        "resolve_abbreviations": True,
        "linker_name": "mesh"   # lighter than UMLS
    }
)

print("Linker loaded")

Loading SciSpacy linker...


/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


https://ai2-s2-scispacy.s3-us-west-2.amazonaws.com/data/linkers/2023-04-23/mesh/tfidf_vectors_sparse.npz not found in cache, downloading to /tmp/tmpg414w9ji


100%|██████████| 68.1M/68.1M [00:03<00:00, 21.5MiB/s]


Finished download, copying /tmp/tmpg414w9ji to cache at /root/.scispacy/datasets/0acb1f67e1908d2211efb5291880a946e905e1a14a87c10cfc640d0711f914c7.e4877c46bb5a882e9729b6abe799b33f195067557a3c0c15086a50471f29b985.tfidf_vectors_sparse.npz
https://ai2-s2-scispacy.s3-us-west-2.amazonaws.com/data/linkers/2023-04-23/mesh/nmslib_index.bin not found in cache, downloading to /tmp/tmpseq49yj8


...

Finished download, copying /tmp/tmpg1f9xq9j to cache at /root/.scispacy/datasets/6a0e66a77d89d942876d1b853abf461e6b16edaef65bebfa7f3a8dd99ff6553b.6eb17e1805a69a55fc151aa59fe42343d2b4be81405127043fd065bf5f9620e0.tfidf_vectorizer.joblib
https://ai2-s2-scispacy.s3-us-west-2.amazonaws.com/data/linkers/2023-04-23/mesh/concept_aliases.json not found in cache, downloading to /tmp/tmpn579chp8


100%|██████████| 29.3M/29.3M [00:00<00:00, 38.8MiB/s]


Finished download, copying /tmp/tmpn579chp8 to cache at /root/.scispacy/datasets/ccb3a55e3a37984902cc7de591d37d56d90eb0962d128536512b8d1219e71bcb.89e92a904a5ccc051bcba6ee26c5744e183dee7197cc835cfeb152b330b44559.concept_aliases.json
https://ai2-s2-scispacy.s3-us-west-2.amazonaws.com/data/kbs/2023-04-23/umls_mesh_2022.jsonl not found in cache, downloading to /tmp/tmp8e12igm4


100%|██████████| 76.4M/76.4M [00:02<00:00, 38.3MiB/s]


Finished download, copying /tmp/tmp8e12igm4 to cache at /root/.scispacy/datasets/5541a1df25533cfafec1fdcf0446c761f998591519c8ad4a73876f48d7e0a224.c4d6e393746f18aaf6eafff94fe1782cebf29ef535b101501e66f1e3462cdb09.umls_mesh_2022.jsonl
Linker loaded


In [7]:
# ============================================================
# SECTION 7 — Normalize Helper
# ============================================================

def normalize(text):

    text = text.lower()

    text = re.sub(
        r"[^a-z0-9\- ]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [8]:
# ============================================================
# SECTION 8 — Link Entity
# ============================================================

LINK_SCORE_THRESHOLD = 0.80

def link_entity(entity_text):

    doc = linker_nlp(entity_text)

    linked_entities = []

    for ent in doc.ents:

        if len(ent._.kb_ents) == 0:
            continue

        # top candidate only
        top_candidate = ent._.kb_ents[0]

        kb_id = top_candidate[0]
        score = top_candidate[1]

        # confidence filter
        if score < LINK_SCORE_THRESHOLD:
            continue

        kb_entry = linker_nlp.get_pipe(
            "scispacy_linker"
        ).kb.cui_to_entity[kb_id]

        linked_entities.append({
            "kb_id": kb_id,
            "canonical_name": kb_entry.canonical_name,
            "link_score": score
        })

    return linked_entities

In [9]:
# ============================================================
# SECTION 9 — Run Entity Linking
# ============================================================

linked_paper_entities = defaultdict(list)

total_entities = 0
linked_entities = 0

for pmid, ents in paper_entities.items():

    for ent in ents:

        total_entities += 1

        mention = ent["entity"]

        # ----------------------------------------------------
        # Skip ML entities (optional)
        # ----------------------------------------------------

        if ent["label"] == "ML_METHOD":

            linked_paper_entities[pmid].append({
                "mention": mention,
                "entity": normalize(mention),
                "canonical_name": None,
                "kb_id": None,
                "label": ent["label"],
                "link_score": None
            })

            continue

        # ----------------------------------------------------
        # Biomedical Linking
        # ----------------------------------------------------

        linked = link_entity(mention)

        # ----------------------------------------------------
        # Link success
        # ----------------------------------------------------

        if len(linked) > 0:

            top_link = linked[0]

            linked_entities += 1

            linked_paper_entities[pmid].append({
                "mention": mention,
                "entity": normalize(
                    top_link["canonical_name"]
                ),
                "canonical_name": top_link["canonical_name"],
                "kb_id": top_link["kb_id"],
                "label": ent["label"],
                "link_score": float(
                    top_link["link_score"]
                )
            })

        # ----------------------------------------------------
        # Link failure fallback
        # ----------------------------------------------------

        else:

            linked_paper_entities[pmid].append({
                "mention": mention,
                "entity": normalize(mention),
                "canonical_name": None,
                "kb_id": None,
                "label": ent["label"],
                "link_score": None
            })

In [10]:
# ============================================================
# SECTION 10 — Stats
# ============================================================

print("\n===================================")
print("LINKING COMPLETE")
print("===================================")

print("Total entities:", total_entities)
print("Linked entities:", linked_entities)

if total_entities > 0:

    print(
        "Link rate:",
        round(
            100 * linked_entities / total_entities,
            2
        ),
        "%"
    )


LINKING COMPLETE
Total entities: 7772
Linked entities: 3774
Link rate: 48.56 %


In [11]:
# ============================================================
# SECTION 11 — Save Output
# ============================================================

with open(OUTPUT_PATH, "w") as f:
    json.dump(
        linked_paper_entities,
        f,
        indent=2
    )

print("\nSaved linked entities to:")
print(OUTPUT_PATH)


Saved linked entities to:
/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/data/processed/clean_papers_with_linked_entities.json


In [12]:
# ============================================================
# SECTION 12 — Example Inspection
# ============================================================

sample_pmid = list(linked_paper_entities.keys())[0]

print("\n===================================")
print("SAMPLE OUTPUT")
print("===================================")

print("PMID:", sample_pmid)

for item in linked_paper_entities[sample_pmid][:10]:

    print("\n-------------------")

    for k, v in item.items():
        print(f"{k}: {v}")


SAMPLE OUTPUT
PMID: 42104431_BACKGROUND

-------------------
mention: breast cancer
entity: breast malignant neoplasms
canonical_name: Breast Malignant Neoplasms
kb_id: C0006142
label: DISEASE
link_score: 0.9906210899353027

-------------------
mention: death
entity: death
canonical_name: None
kb_id: None
label: DISEASE
link_score: None

-------------------
mention: tumor
entity: tumor
canonical_name: Tumor
kb_id: C0027651
label: DISEASE
link_score: 0.9929314851760864

-------------------
mention: alns
entity: alns
canonical_name: None
kb_id: None
label: DISEASE
link_score: None

-------------------
mention: aln
entity: aln
canonical_name: None
kb_id: None
label: CHEMICAL
link_score: None

-------------------
mention: breast cancer
entity: breast malignant neoplasms
canonical_name: Breast Malignant Neoplasms
kb_id: C0006142
label: CANCER
link_score: 0.9906210899353027

-------------------
mention: women
entity: woman
canonical_name: Woman
kb_id: C0043210
label: ORGANISM
link_score: 0.